# A3-M2 · KMeans
교안의 같은 번호 단원과 함께 봅니다.

## 04-1. 준비 — 눈으로 보이는 2차원 점 12개

In [1]:
import faiss
import numpy as np

x = np.array([[1, 1], [1, 2], [2, 1], [2, 2],      # 0~3: 왼쪽 아래 묶음
              [8, 8], [8, 9], [9, 8], [9, 9],      # 4~7: 오른쪽 위 묶음
              [1, 8], [1, 9], [2, 8], [2, 9]],     # 8~11: 왼쪽 위 묶음
             dtype=np.float32)
print(x.shape)

(12, 2)


## 04-2. K-means를 numpy로 한 단계씩 — FAISS가 안에서 하는 일

In [2]:
def kmeans_step_by_step(x, init_idx, n_iter=10):
    x = x.astype(np.float64)
    C = x[init_idx].copy()                                     # 시작 중심 = 고른 점들
    print("시작 중심:", C.tolist())
    for it in range(1, n_iter + 1):
        d2 = ((x[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)  # [12, 3] 점마다 세 중심까지 제곱거리
        assign = d2.argmin(axis=1)                              # ① 배정: 가장 가까운 중심 번호
        obj = d2[np.arange(len(x)), assign].sum()              # 오차 = 자기 중심까지 거리의 합
        newC = np.array([x[assign == c].mean(axis=0) for c in range(len(C))])   # ② 갱신: 평균 위치로
        print(f"반복 {it}: 소속 {assign}  오차 {obj:6.2f}  → 새 중심 {newC.round(2).tolist()}")
        if np.allclose(newC, C):
            print("   중심이 그대로 → 수렴(멈춤)")
            break
        C = newC
    return C, assign

print("[A] 시작 중심을 0, 1, 8번 점으로")
kmeans_step_by_step(x, [0, 1, 8])
print("\n[B] 시작 중심을 0, 4, 8번 점으로")
kmeans_step_by_step(x, [0, 4, 8])

[A] 시작 중심을 0, 1, 8번 점으로
시작 중심: [[1.0, 1.0], [1.0, 2.0], [1.0, 8.0]]
반복 1: 소속 [0 1 0 1 2 2 2 2 2 2 2 2]  오차 234.00  → 새 중심 [[1.5, 1.0], [1.5, 2.0], [5.0, 8.5]]
반복 2: 소속 [0 1 0 1 2 2 2 2 2 2 2 2]  오차 103.00  → 새 중심 [[1.5, 1.0], [1.5, 2.0], [5.0, 8.5]]
   중심이 그대로 → 수렴(멈춤)

[B] 시작 중심을 0, 4, 8번 점으로
시작 중심: [[1.0, 1.0], [8.0, 8.0], [1.0, 8.0]]
반복 1: 소속 [0 0 0 0 1 1 1 1 2 2 2 2]  오차  12.00  → 새 중심 [[1.5, 1.5], [8.5, 8.5], [1.5, 8.5]]
반복 2: 소속 [0 0 0 0 1 1 1 1 2 2 2 2]  오차   6.00  → 새 중심 [[1.5, 1.5], [8.5, 8.5], [1.5, 8.5]]
   중심이 그대로 → 수렴(멈춤)


(array([[1.5, 1.5],
        [8.5, 8.5],
        [1.5, 8.5]]),
 array([0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]))

## 04-4. faiss.Kmeans 기본 문법

In [3]:
kmeans = faiss.Kmeans(d=2, k=3, niter=20, seed=42)
kmeans.train(x)

print("중심점:\n", kmeans.centroids)
print("반복별 오차(obj):", kmeans.obj)

D, I = kmeans.index.search(x, 1)       # 각 점에서 가장 가까운 중심 1개
print("소속:", I.ravel())
print("중심까지 거리:", D.ravel())

중심점:
 [[1.5 5. ]
 [8.5 8. ]
 [8.5 9. ]]
반복별 오차(obj): [513.         121.44000244 103.         103.        ]
소속: [0 0 0 0 1 2 1 2 0 0 0 0]
중심까지 거리: [16.25  9.25 16.25  9.25  0.25  0.25  0.25  0.25  9.25 16.25  9.25 16.25]


## 04-5. 초기값 문제와 nredo

In [4]:
for seed in [1, 2, 3, 42, 123]:
    km = faiss.Kmeans(2, 3, niter=20, seed=seed, min_points_per_centroid=1)
    km.train(x)
    print(f"seed={seed:<4} 최종 오차={km.obj[-1]:7.2f} 소속={km.index.search(x, 1)[1].ravel()}")

best = faiss.Kmeans(2, 3, niter=20, seed=42, nredo=5, min_points_per_centroid=1)
best.train(x)
print("nredo=5 → 오차", best.obj[-1], "소속", best.index.search(x, 1)[1].ravel())
print("중심점:\n", best.centroids)

seed=1    최종 오차= 103.33 소속=[0 0 0 2 1 1 1 1 1 1 1 1]
seed=2    최종 오차=   6.00 소속=[2 2 2 2 0 0 0 0 1 1 1 1]
seed=3    최종 오차= 103.00 소속=[1 2 1 2 0 0 0 0 0 0 0 0]
seed=42   최종 오차= 103.00 소속=[0 0 0 0 1 2 1 2 0 0 0 0]
seed=123  최종 오차= 102.00 소속=[1 1 0 0 2 2 2 2 1 1 0 0]
nredo=5 → 오차 6.0 소속 [1 1 1 1 0 0 0 0 2 2 2 2]
중심점:
 [[8.5 8.5]
 [1.5 1.5]
 [1.5 8.5]]


## 04-6. 새 점 배정 · 반복 기록 · 코사인용 spherical

In [5]:
new_point = np.array([[1.5, 8.5], [8.2, 8.7]], dtype=np.float32)
D, I = best.index.search(new_point, 1)
print("새 점의 소속:", I.ravel(), "거리:", D.ravel())

last = best.iteration_stats[-1]
print("마지막 반복 기록 키:", list(last.keys()))
print("학습 데이터 조건:", best.cp.min_points_per_centroid, "~", best.cp.max_points_per_centroid, "개/중심")

sph = faiss.Kmeans(2, 3, niter=20, seed=1, nredo=5, spherical=True, min_points_per_centroid=1)
sph.train(x)
print("spherical 중심점 길이:", np.linalg.norm(sph.centroids, axis=1).round(3))

새 점의 소속: [2 0] 거리: [0.         0.13000004]
마지막 반복 기록 키: ['obj', 'time', 'time_search', 'imbalance_factor', 'nsplit']
학습 데이터 조건: 1 ~ 256 개/중심
spherical 중심점 길이: [1. 1. 1.]
